In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_dataset.csv to cleaned_dataset (1).csv


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('cleaned_dataset.csv')

def combine_text(row):
    parts = []
    for col in ['product_name', 'description', 'gender', 'occasion', 'color']:
        if col in row and pd.notna(row[col]):
            parts.append(str(row[col]))
    return ' '.join(parts)

# If the columns exist, create search_text
df['search_text'] = df.apply(combine_text, axis=1)

# 4. Now, load SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 5. Encode product texts
product_embeddings = model.encode(df['search_text'].tolist(), show_progress_bar=True)

# 6. Search function
def search_products(query, top_k=5):
    query_embedding = model.encode([query])
    similarity_scores = cosine_similarity(query_embedding, product_embeddings).flatten()
    top_indices = similarity_scores.argsort()[-top_k:][::-1]

    results = df.iloc[top_indices]

    for idx, row in results.iterrows():
        print(f"🔎 Product {idx+1}:")
        print(f"Title      : {row.get('product_name', 'N/A')}")
        print(f"Brand      : {row.get('brand', 'N/A')}")
        print(f"Color      : {row.get('color', 'N/A')}")
        print(f"Price      : ₹{row.get('selling_price', 'N/A')} (Original ₹{row.get('actual_price', 'N/A')})")
        print(f"Link       : {row.get('product_link', 'N/A')}")
        print(f"Image Link : {row.get('image_link', 'N/A')}")
        print("-" * 70)

    return results[['product_name', 'brand', 'selling_price', 'actual_price', 'product_link', 'image_link']]

Batches:   0%|          | 0/1407 [00:00<?, ?it/s]

In [ ]:
query = "blue tie"
results = search_products(query)

🔎 Product 15161:
Title      : Solid V Neck Casual Men Blue Sweater
Brand      : Man
Color      : Blue
Price      : ₹499 (Original ₹2199)
Link       : https://www.flipkart.com/manra-solid-v-neck-casual-men-blue-sweater/p/itmfbf3b216ac996?pid=SWTFYHPVHZSVMHYR&lid=LSTSWTFYHPVHZSVMHYRMXOAHS&marketplace=FLIPKART&srno=b_1_10&otracker=browse&fm=organic&iid=0f3b6d48-c1ad-4725-8feb-92f5d8321831.SWTFYHPVHZSVMHYR.SEARCH&ssid=14u56zojjk0000001612113908106
Image Link : https://rukminim1.flixcart.com/image/128/128/kirr24w0-0/sweater/n/b/j/l-men-closed-zipper-blue-103-4-manra-original-imafyhpvxf6bx997.jpeg?q=70
----------------------------------------------------------------------
🔎 Product 15133:
Title      : nu-Lite Satin Tie Pin Set  (Blue)
Brand      : Unknown
Color      : Unknown
Price      : ₹349 (Original ₹1539)
Link       : https://www.flipkart.com/nu-lite-satin-tie-pin-set/p/itm4e2abf68050bd?pid=CTPFVPHAGX8YA7RD&lid=LSTCTPFVPHAGX8YA7RD1JGUOO&marketplace=FLIPKART&srno=b_3_98&otracker=browse&f

In [37]:
import pickle

# Save embeddings
with open('/content/product_embeddings.pkl', 'wb') as f:
    pickle.dump(product_embeddings, f)

# Also save the corresponding DataFrame
df.to_csv('/content/cleaned_products_with_search_text.csv', index=False)
